In [1]:
# CELL 1: Install Required Libraries
# We force upgrade sympy and torchao to resolve recent Colab version conflicts
!pip install -q --upgrade sympy torchao
!pip install -q transformers peft pandas Pillow torchvision tqdm
print("✅ Libraries installed successfully!")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Charlene C. Dilig\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


✅ Libraries installed successfully!



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Charlene C. Dilig\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
# CELL 2: Mount Drive and Define Transforms
import torchvision.transforms as T
import torch

# 2. Define the Augmentation for TRAINING (Random Erasing + Advanced Color Jitter)
train_transform = T.Compose([
    T.Resize((224, 224)),                    # Standardize size for FG-CLIP
    T.RandomHorizontalFlip(p=0.5),           # 50% chance to flip
    # --- UPGRADED: Full Spectrum Color Jitter ---
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.4, hue=0.1),
    # --------------------------------------------
    T.ToTensor(),                            # Convert to Tensor for Erasing
    T.RandomErasing(p=0.3, scale=(0.02, 0.15)), # 30% chance to mask a feature!
    T.ToPILImage()                           # Convert back for the HF Processor
])

# 3. Define the rule for VALIDATION (No Augmentation, pristine images only)
eval_transform = T.Compose([
    T.Resize((224, 224))
])

print("✅ V2 Augmentations defined!")

✅ V2 Augmentations defined!


In [3]:
# CELL 3: The PyTorch Dataset Class & HARD NEGATIVE MINING SAMPLER
import pandas as pd
import os
import random
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

class LostAndFoundDataset(Dataset):
    def __init__(self, csv_file, image_dir, split_type, transform=None):
        full_data = pd.read_csv(csv_file)
        self.data = full_data[full_data['split'] == split_type].reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        lost_img_path = os.path.join(self.image_dir, row['lost_image'])
        lost_image = Image.open(lost_img_path).convert('RGB')

        if self.transform:
            lost_image = self.transform(lost_image)

        return {
            'lost_image': lost_image,
            'positive_caption': row['positive_caption'],
            'hard_negative_caption': row['hard_negative_caption']
        }

def custom_collate(batch):
    return {
        'lost_image': [item['lost_image'] for item in batch],
        'positive_caption': [item['positive_caption'] for item in batch],
        'hard_negative_caption': [item['hard_negative_caption'] for item in batch]
    }

# ==========================================
# 🚨 NEW: THE HARD NEGATIVE MINING SAMPLER 🚨
# ==========================================
class CategoryBatchSampler(Sampler):
    def __init__(self, dataset_df, batch_size):
        self.batch_size = batch_size

        # Group all item indices by their category (the word before the '_')
        self.category_to_indices = defaultdict(list)
        for idx, row in dataset_df.iterrows():
            filename = row['lost_image']
            category = filename.split('_')[0]
            self.category_to_indices[category].append(idx)

    def __iter__(self):
        batches = []
        for cat, indices in self.category_to_indices.items():
            random.shuffle(indices)
            for i in range(0, len(indices), self.batch_size):
                chunk = indices[i:i + self.batch_size]
                if len(chunk) > 1:
                    batches.append(chunk)

        random.shuffle(batches)
        for batch in batches:
            yield batch

    def __len__(self):
        return sum((len(indices) // self.batch_size) + (1 if len(indices) % self.batch_size > 1 else 0)
                   for indices in self.category_to_indices.values())

# ==========================================
# INITIALIZATION
# ==========================================
CSV_PATH = '../captions/dataset_captionsV2.csv'
IMG_DIR = '../images'

# Load the Datasets
train_dataset = LostAndFoundDataset(CSV_PATH, IMG_DIR, split_type='train', transform=train_transform)
val_dataset = LostAndFoundDataset(CSV_PATH, IMG_DIR, split_type='val', transform=eval_transform)

# 🚨 THE MAGIC SWITCH: Attach the Hard Negative Sampler 🚨
hn_sampler = CategoryBatchSampler(train_dataset.data, batch_size=8)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=hn_sampler, # <-- Replaces shuffle=True
    collate_fn=custom_collate
)

val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=custom_collate)

print(f"✅ Loaded {len(train_dataset)} Training Pairs using Category-Level Hard Negative Mining!")
print(f"✅ Loaded {len(val_dataset)} Validation Pairs.")

✅ Loaded 420 Training Pairs using Category-Level Hard Negative Mining!
✅ Loaded 90 Validation Pairs.


In [4]:
# CELL 4: Model Architecture & LoRA
from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model

model_id = "qihoo360/fg-clip-base"
print(f"Downloading Base Model: {model_id}...")

processor = CLIPProcessor.from_pretrained(model_id)
base_model = CLIPModel.from_pretrained(model_id)

# Configure LoRA strictly for ALL Attention Layers (Goldilocks r=20)
lora_config = LoraConfig(
    r=20,                  # Reduced from 24 (V5: more regularization, push convergence later)
    lora_alpha=40,         # Matched to r=20
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"], 
    lora_dropout=0.1,      
    bias="none"
)

# Inject the adapters
lora_model = get_peft_model(base_model, lora_config)

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
lora_model.to(device)

print("\n✅ Model Ready! Printing trainable parameters for your paper:")
lora_model.print_trainable_parameters()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



✅ Model Ready! Printing trainable parameters for your paper:
trainable params: 2,457,600 || all params: 152,078,337 || trainable%: 1.6160


In [5]:
print("============================================================")
print("🔎 PRE-TRAINING MODEL ARCHITECTURE INSPECTION")
print("============================================================")

try:
    # We reach directly into the PyTorch architecture of the loaded base_model
    # to find the convolutional layer responsible for chopping the image.
    patch_layer = base_model.vision_model.embeddings.patch_embedding

    patch_size = patch_layer.kernel_size[0]
    image_size = 224 # Standard CLIP input size
    num_patches = (image_size // patch_size) ** 2

    print(f"Target Layer: {patch_layer}")
    print(f"Kernel Size:  {patch_layer.kernel_size}  <-- This is your Patch Size!")
    print(f"Stride:       {patch_layer.stride}")

    print("\n--- Physical Grid Calculation ---")
    print(f"Math: {image_size} / {patch_size} = {image_size // patch_size}")
    print(f"Grid: {image_size // patch_size} x {image_size // patch_size} = {num_patches} total patches.")
    print(f"Tensor Shape: [1, {num_patches + 1}, 768] (Includes +1 for the CLS Token)")

    print("\n============================================================")
    print("🎯 FINAL VERDICT")
    print("============================================================")
    if patch_size == 16:
        print("✅ CONFIRMED: You are officially fine-tuning a ViT-B/16 model.")
    elif patch_size == 32:
        print("❌ WARNING: This is a ViT-B/32 model. Update your methodology!")
    else:
        print(f"⚠️ UNKNOWN: Patch size is {patch_size}.")

except AttributeError:
    print("⚠️ Error: Could not locate the patch_embedding layer. Ensure your variable is named 'base_model'.")

🔎 PRE-TRAINING MODEL ARCHITECTURE INSPECTION
Target Layer: Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
Kernel Size:  (16, 16)  <-- This is your Patch Size!
Stride:       (16, 16)

--- Physical Grid Calculation ---
Math: 224 / 16 = 14
Grid: 14 x 14 = 196 total patches.
Tensor Shape: [1, 197, 768] (Includes +1 for the CLS Token)

🎯 FINAL VERDICT
✅ CONFIRMED: You are officially fine-tuning a ViT-B/16 model.


In [6]:
# CELL 5: The Optimized Training Loop (Adjusted for 420 Training Pairs)
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup

# --- 1. DEFINE LOSS & OPTIMIZER (The Fix) ---
# Ensuring loss_fn is explicitly defined here
loss_fn = nn.CrossEntropyLoss() 

# Optimized LR for small datasets (2e-5 is slower than 3e-5 to push convergence past epoch 15)
optimizer = AdamW(filter(lambda p: p.requires_grad, lora_model.parameters()), 
                  lr=2e-5, 
                  weight_decay=0.1) # Added weight decay for regularization

num_epochs = 20
total_steps = len(train_loader) * num_epochs
# Increased warmup to 20% to stabilize gradients on a small batch
warmup_steps = int(0.20 * total_steps) 

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# --- 2. HYPERPARAMETER OVERRIDES ---
# Increased from 0.02 to 0.05 to prevent overfitting on hard negatives
STRICT_TEMP = 0.05 

print(f"🔧 Scheduler activated: {total_steps} total steps, {warmup_steps} warmup steps.")
print(f"🌡️ Temperature set to {STRICT_TEMP} for better generalization.")

# --- 3. EARLY STOPPING SETUP ---
best_val_loss = float('inf')
best_recall10 = 0.0
patience = 15 
patience_counter = 0
save_path = "../lorafinetuned/fgclip-lora-finetunedV5-1200-HNM"

# --- 4. TRAINING LOOP ---
for epoch in range(num_epochs):
    lora_model.train()
    total_train_loss = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for batch in train_bar:
        optimizer.zero_grad()

        images = batch['lost_image']
        pos_texts = batch['positive_caption']
        neg_texts = batch['hard_negative_caption']
        all_texts = pos_texts + neg_texts # [Batch Positives + Batch Negatives]

        inputs = processor(text=all_texts, images=images, return_tensors="pt", padding=True).to(device)
        outputs = lora_model(**inputs)

        # Normalize embeddings for cosine similarity
        image_embeds = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
        text_embeds = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)

        # Calculate Logits: Shape [Batch, 2*Batch]
        custom_logits = (image_embeds @ text_embeds.T) / STRICT_TEMP

        # Targets are indices 0 to (Batch-1), matching the positive captions
        labels = torch.arange(len(images)).to(device)
        loss = loss_fn(custom_logits, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()
        train_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)
    # --- VALIDATION PHASE ---
    lora_model.eval()
    total_val_loss = 0
    # Collect embeddings for Recall@K
    all_val_image_embeds = []
    all_val_text_embeds = []
    with torch.no_grad():
        for batch in val_loader:
            images = batch['lost_image']
            pos_texts = batch['positive_caption']
            neg_texts = batch['hard_negative_caption']

            inputs = processor(text=pos_texts + neg_texts, images=images, return_tensors="pt", padding=True).to(device)
            outputs = lora_model(**inputs)

            image_embeds = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
            text_embeds = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)

            custom_logits = (image_embeds @ text_embeds.T) / STRICT_TEMP
            labels = torch.arange(len(images)).to(device)

            loss = loss_fn(custom_logits, labels)
            total_val_loss += loss.item()

            # Collect embeddings for Recall@K (only positive texts, one per image)
            n = len(images)
            all_val_image_embeds.append(image_embeds[:n])
            all_val_text_embeds.append(text_embeds[:n])

    avg_val_loss = total_val_loss / len(val_loader)

    # ---- Compute Recall@K ----
    val_img_embeds = torch.cat(all_val_image_embeds, dim=0)
    val_txt_embeds = torch.cat(all_val_text_embeds, dim=0)
    n_val = val_img_embeds.shape[0]
    sim_matrix = val_img_embeds @ val_txt_embeds.T  # [n_val, n_val]
    ranks = (-sim_matrix).argsort(dim=1)  # descending similarity
    targets = torch.arange(n_val).to(device)
    recalls = {}
    for k in [1, 5, 10]:
        recalls[f"recall@{k}"] = (ranks[:, :k] == targets.unsqueeze(1)).any(dim=1).float().mean().item()
    print(f"\n=> Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | R@1: {recalls['recall@1']:.4f} | R@5: {recalls['recall@5']:.4f} | R@10: {recalls['recall@10']:.4f}")

    # Save checkpoint if val loss improves OR if recall@10 improves
    improved = False
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        improved = True
        lora_model.save_pretrained(save_path)
        print("\u0001f7e2 Improvement (val loss)! Best model saved.\n")
    # Also save if recall@10 hits a new best (different from val loss checkpoint)
    if recalls["recall@10"] > best_recall10:
        best_recall10 = recalls["recall@10"]
        if not improved:
            lora_model.save_pretrained(save_path)
            print("\u0001f7e2 Improvement (recall@10)! Best model saved.\n")
        patience_counter = 0
    if not improved and recalls["recall@10"] <= best_recall10:
        patience_counter += 1
        print(f"\u0001f7e1 No improvement. Patience: {patience_counter}/{patience}\n")
        if patience_counter >= patience:
            print(f"\u0001f7db EARLY STOPPING! Best Val Loss: {best_val_loss:.4f}, Best R@10: {best_recall10:.4f}")
            break


🔧 Scheduler activated: 1060 total steps, 212 warmup steps.
🌡️ Temperature set to 0.05 for better generalization.


Epoch 1/20 [Train]: 100%|██████████| 53/53 [00:44<00:00,  1.20it/s, loss=0.7494]



=> Epoch 1 | Train Loss: 0.9499 | Val Loss: 1.2179 | R@1: 0.8222 | R@5: 0.9222 | R@10: 0.9556
f7e2 Improvement (val loss)! Best model saved.



Epoch 2/20 [Train]: 100%|██████████| 53/53 [00:42<00:00,  1.25it/s, loss=0.7629]



=> Epoch 2 | Train Loss: 0.9250 | Val Loss: 1.2056 | R@1: 0.8222 | R@5: 0.9222 | R@10: 0.9556
f7e2 Improvement (val loss)! Best model saved.



...


=> Epoch 18 | Train Loss: 0.4076 | Val Loss: 1.0175 | R@1: 0.8111 | R@5: 0.9333 | R@10: 0.9556
f7e1 No improvement. Patience: 1/15



Epoch 19/20 [Train]: 100%|██████████| 53/53 [00:42<00:00,  1.24it/s, loss=0.5282]



=> Epoch 19 | Train Loss: 0.4438 | Val Loss: 1.0177 | R@1: 0.8111 | R@5: 0.9333 | R@10: 0.9556
f7e1 No improvement. Patience: 2/15



Epoch 20/20 [Train]: 100%|██████████| 53/53 [00:42<00:00,  1.24it/s, loss=0.5298]



=> Epoch 20 | Train Loss: 0.4353 | Val Loss: 1.0177 | R@1: 0.8111 | R@5: 0.9333 | R@10: 0.9556
f7e1 No improvement. Patience: 3/15



In [7]:
# 📊 CELL: The Exhaustive "Final Boss" Experiment Summary
def print_final_boss_summary():
    import torch, transformers, peft, platform
    
    out = []
    out.append("=" * 80)
    out.append("🔬 FINAL BOSS: EXHAUSTIVE EXPERIMENT CONFIGURATION & RESULTS SUMMARY")
    out.append("=" * 80)
    
    # 1. Hardware & Environment
    out.append("\n[1] HARDWARE & ENVIRONMENT")
    try: 
        out.append(f"    🖥️  OS:              {platform.system()} {platform.release()}")
        out.append(f"    🐍 Python Version:  {platform.python_version()}")
        out.append(f"    📦 PyTorch Version: {torch.__version__}")
        out.append(f"    📦 Transformers:    {transformers.__version__}")
        out.append(f"    📦 PEFT Version:    {peft.__version__}")
        if torch.cuda.is_available():
            out.append(f"    🚀 GPU:             {torch.cuda.get_device_name(0)}")
            mem_used = torch.cuda.max_memory_allocated() / (1024 ** 3)
            out.append(f"    💾 Max VRAM Used:   {mem_used:.2f} GB")
        else:
            out.append("    🐌 GPU:             None (CPU Training)")
        out.append(f"    🎲 Global Seed:     {torch.initial_seed()}")
    except Exception as e: out.append(f"    ⚠️  Error: {e}")

    # 2. Base Model & Dataset
    out.append("\n[2] MODEL & DATASET")
    try: out.append(f"    🤖 Base Model ID:       {model_id}")
    except NameError: pass
    try: out.append(f"    📏 Max Text Length:     {processor.tokenizer.model_max_length} tokens")
    except NameError: pass
    try: out.append(f"    🔤 Text Padding:        True (Longest in batch)")
    except Exception: pass
    try:
        out.append(f"    🖼️  Train Images:        {len(train_dataset)}")
        out.append(f"    🖼️  Validation Images:   {len(val_dataset)}")
    except NameError: pass

    try:
        if 'hn_sampler' in globals():
            out.append(f"    ⛏️  Mining Strategy:     Hard Negative Category Mining")
        else:
            out.append(f"    ⛏️  Mining Strategy:     Standard Random Shuffle")
    except Exception: pass

    # 3. Model Architecture Details
    out.append("\n[3] ARCHITECTURE & LoRA")
    try:
        trainable, total = lora_model.get_nb_trainable_parameters()
        out.append(f"    📊 Trainable Params:    {trainable:,} ({(trainable/total)*100:.4f}% of total)")
        out.append(f"    🧠 LoRA Rank (r):       {lora_config.r}")
        out.append(f"    🧠 LoRA Alpha:          {lora_config.lora_alpha}")
        out.append(f"    🎯 Target Modules:      {lora_config.target_modules}")
        out.append(f"    💧 Dropout:             {lora_config.lora_dropout}")
        out.append(f"    🧮 Model dtype:         {lora_model.dtype}")
    except NameError: pass

    # 4. Hyperparameters & Optimizer
    out.append("\n[4] HYPERPARAMETERS & TRAINING")
    try: 
        pg = optimizer.param_groups[0]
        out.append(f"    ⚙️  Optimizer:           {type(optimizer).__name__}")
        
        # Check if the scheduler recorded the initial LR
        if 'initial_lr' in pg:
            out.append(f"    📉 Start Learn Rate:    {pg['initial_lr']}")
        else:
            # If no scheduler (like V1), the LR never changed
            out.append(f"    📉 Start Learn Rate:    {pg['lr']}")
            
        out.append(f"    📉 End Learn Rate:      {pg['lr']}")
        
        out.append(f"    ⚓ Weight Decay:        {pg.get('weight_decay', 0.0)}")
        out.append(f"    🎢 Betas (B1, B2):      {pg.get('betas', 'N/A')}")
        out.append(f"    🔢 Optimizer Epsilon:   {pg.get('eps', 'N/A')}")
    except NameError: pass
    try: out.append(f"    ⚖️  Loss Function:       {type(loss_fn).__name__}")
    except NameError: pass
    
    # Advanced Temperature Catch (Calculates actual float temperature)
    try: 
        if 'STRICT_TEMP' in globals():
            out.append(f"    🌡️  Temperature:         {STRICT_TEMP} (Manual Strict Override)")
        else:
            actual_temp = 1.0 / lora_model.base_model.logit_scale.exp().item()
            out.append(f"    🌡️  Temperature:         {actual_temp:.4f} (Auto-Scaled by HF Logit Parameter)")
    except Exception: pass

    try: 
        out.append(f"    📦 Train Batch Size:    {train_loader.batch_size}")
        out.append(f"    📦 Val Batch Size:      {val_loader.batch_size}")
        out.append(f"    👷 Dataloader Workers:  {train_loader.num_workers}")
    except NameError: pass
    
    try: 
        out.append(f"    📈 Total Steps:         {total_steps}")
        if 'warmup_steps' in globals():
            out.append(f"    🔥 Warmup Steps:        {warmup_steps} (Cosine Scheduler)")
    except NameError: pass

    # 5. Training Results
    out.append("\n[5] TRAINING RESULTS")
    try: out.append(f"    📉 Train Loss (at Stop): {avg_train_loss:.4f}")
    except NameError: pass
    try: out.append(f"    📉 Val Loss (at Stop):   {avg_val_loss:.4f}")
    except NameError: pass
    try: out.append(f"    🏆 Best Val Loss:        {best_val_loss:.4f} (Model Saved Here!)")
    except NameError: pass
    try: out.append(f"    🏁 Stopped at Epoch:     {epoch+1} / {num_epochs}")
    except NameError: pass
    try: out.append(f"    🛑 Early Stop Patience:  {patience} epochs")
    except NameError: pass


    # 6. Output Details
    out.append("\n[6] OUTPUT")
    try: out.append(f"    💾 Saved To:            {save_path}")
    except NameError: pass

    # 7. Augmentations
    out.append("\n[7] DATA AUGMENTATIONS")
    try:
        out.append(str(train_transform))
    except NameError:
        out.append("    ⚠️  Could not find train_transform variable.")
        
    out.append("=" * 80)
        # === NEW: PRINT AND SAVE TO FILE ===
    report_text = "\n".join(out)
    print(report_text)

    try:
        import os
        if 'save_path' in globals():
            # Saves it right next to your weights!
            txt_path = os.path.join(save_path, "training_summary_report.txt")
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(report_text)
            print(f"\n💾 Training Summary safely saved to: {txt_path}")
    except Exception as e:
        print(f"\n⚠️ Could not save text file: {e}")

# Run the function
print_final_boss_summary()



🔬 FINAL BOSS: EXHAUSTIVE EXPERIMENT CONFIGURATION & RESULTS SUMMARY

[1] HARDWARE & ENVIRONMENT
    🖥️  OS:              Windows 10
    🐍 Python Version:  3.13.3
    📦 PyTorch Version: 2.6.0+cu124
    📦 Transformers:    4.57.1
    📦 PEFT Version:    0.19.1
    🚀 GPU:             NVIDIA GeForce RTX 3050 Laptop GPU
    💾 Max VRAM Used:   2.19 GB
    🎲 Global Seed:     10443617984700

[2] MODEL & DATASET
    🤖 Base Model ID:       qihoo360/fg-clip-base
    📏 Max Text Length:     77 tokens
    🔤 Text Padding:        True (Longest in batch)
    🖼️  Train Images:        420
    🖼️  Validation Images:   90
    ⛏️  Mining Strategy:     Hard Negative Category Mining

[3] ARCHITECTURE & LoRA
    📊 Trainable Params:    2,457,600 (1.6160% of total)
    🧠 LoRA Rank (r):       20
    🧠 LoRA Alpha:          40
    🎯 Target Modules:      {'out_proj', 'k_proj', 'q_proj', 'v_proj'}
    💧 Dropout:             0.1
    🧮 Model dtype:         torch.float32

[4] HYPERPARAMETERS & TRAINING
    ⚙️  Optimizer: 